# GTEx V11 brain-region transcriptomic mapping
**BMEN 600 | Public Google Colab notebook**

**Research question:** How accurately can gene expression in one brain region predict expression in another in unseen GTEx donors, and how does performance vary across region pairs?

This notebook evaluates donor-matched cross-region prediction and descriptive associations between gene-wise predictability and expression characteristics. It uses **12 brain regions only**, with no added disease-gene panels. The map represents statistical predictability, not anatomical connectivity or causal regulation.

**Run in Google Colab:** choose **Runtime → Run all**. Section 2 downloads the existing public data release from `ellastokowski/BMEN600Group23`, checks its SHA-256 digest, and extracts only the 12 named brain matrices. No owner edits or manual data uploads are required. The archive contains an additional spinal-cord file; that file is never extracted, loaded, analyzed, graphed, or exported.

The default run reproduces six pilot directions with 500 source genes, 1,000 target genes, and seed 42. Set `RUN_ALL_PAIRS=True` for the full eligible brain-region comparison; this takes longer.


**Verification scope:** The six brain-only models use 3,030 available tissue records from 385 unique donors across the input dataset; individual paired cohorts contain 194–233 donors. Median gene-wise R² values are approximately 0.450, 0.389, 0.497, 0.365, 0.294, and 0.272 in the configured direction order. Pooled skill is a separate statistic. These scores describe one split and training-selected gene sets.

Execution counters and saved outputs are cleared in this copy so that running it generates its own tables and figures. The full regional map, repeated-split validation, normalization sensitivity, and covariate adjustment remain future work. Molecular-feature correlations below are descriptive, with no independent-gene significance claim.


## 1. Scientific rationale and analysis choices
Regional gene-expression patterns have been mapped in the Allen Human Brain Atlas [1], and reproducible transcriptional signatures have been related to anatomical and cellular organization [2]. GTEx provides a multi-tissue bulk RNA-seq resource [3]. These studies motivate the question; they do not establish that this notebook's prediction model will succeed.

Each independent case is a **donor**, not a gene or a tissue sample. Source and target regions must come from the same donor. A held-out donor's target measurements never enter fitting, feature selection, scaling, or tuning. Ridge regression provides regularization for correlated, high-dimensional predictors [4]; a separate inner validation split selects its penalty without using the test set [5].

We use log2(CPM + 1), calculated per sample from the complete count matrix. This is an exploratory normalization, **not TPM** and not TMM. Library-size scaling may retain compositional bias [6]; compare against TMM or another justified normalization before strong conclusions. These files alone lack age, sex, RIN, postmortem interval and batch metadata. Apparent transferability can reflect these factors or cell-type composition [3]. This analysis cannot establish neurodegenerative mechanisms, disease biomarkers or clinical utility.

BRONTE integrates GTEx with Allen Brain Atlas information to impute expression in additional tissues [7]. This notebook is a simpler pairwise ridge-regression analysis and does not reproduce BRONTE, use genotype data, or perform disease-risk inference. The cited BRONTE version is a preprint. Its imputation validation includes held-out measurements; avoid equating that design with this notebook's unseen-donor evaluation.

The 2020 GTEx paper describes V8 scientific background; Adult GTEx V11 release information is documented separately by the GTEx Portal [8]. The GitHub archive is the team’s redistribution of the supplied matrices. Release and per-file checksums establish which files this notebook used; they do not replace original GTEx download provenance.


In [ ]:
from pathlib import Path
import gzip, json, hashlib, platform, itertools, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from IPython.display import display
from datetime import datetime, timezone
from uuid import uuid4

SEED = 42
RUN_ALL_PAIRS = False
SOURCE = 'caudate_basal_ganglia'
TARGET = 'putamen_basal_ganglia'
MIN_PAIRED_DONORS = 60
N_SOURCE_GENES = 500
N_TARGET_GENES = 1000
ALPHAS = [1., 10., 100., 1000.]
N_BOOTSTRAPS = 1000
BASE_DIR = Path('/content')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:8]
OUT = BASE_DIR / 'gtex_results' / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
ILLUSTRATIVE_PAIRS = [
    ('caudate_basal_ganglia','putamen_basal_ganglia'),
    ('putamen_basal_ganglia','caudate_basal_ganglia'),
    ('cortex','frontal_cortex_ba9'), ('frontal_cortex_ba9','cortex'),
    ('cerebellum','cerebellar_hemisphere'), ('cerebellar_hemisphere','cerebellum')]
BRAIN_REGIONS = ['amygdala', 'anterior_cingulate_cortex_ba24', 'caudate_basal_ganglia', 'cerebellar_hemisphere', 'cerebellum', 'cortex', 'frontal_cortex_ba9', 'hippocampus', 'hypothalamus', 'nucleus_accumbens_basal_ganglia', 'putamen_basal_ganglia', 'substantia_nigra']


## 2. Download and verify the 12 brain matrices
The configured release is [v1.0-data](https://github.com/ellastokowski/BMEN600Group23/releases/tag/v1.0-data), asset **`gene_reads_GTEx.zip`**. Its SHA-256 digest and the 12 brain-file digests are fixed below. A changed or incomplete download stops with an error instead of silently changing the analysis. Re-running may reuse the cached ZIP only if its digest matches.

Only the explicit brain filenames are extracted and passed to the loader. Additional archive members are ignored. If the project intentionally replaces its data release, the maintainers must review and update the recorded checksums.


In [ ]:
from urllib.request import Request, urlopen
import zipfile

GITHUB_OWNER = 'ellastokowski'
GITHUB_REPO = 'BMEN600Group23'
RELEASE_TAG = 'v1.0-data'
ZIP_NAME = 'gene_reads_GTEx.zip'
EXPECTED_ZIP_SHA256 = 'b5e3418a3b880796845a6bd5f7fd01f4856dfc97655bfb4fea6502c8721533b4'
EXPECTED_FILE_SHA256 = {'gene_reads_adult_gtex_v11_brain_amygdala.gct.gz': '4b108088cae871451b59a32b72df10feb0e560ec502de29d7ea2480ec81ad330', 'gene_reads_adult_gtex_v11_brain_anterior_cingulate_cortex_ba24.gct.gz': 'e1e75dc38601dea31c578118e03a4e0cf043286daf5ba88bde811c63be6e6a5c', 'gene_reads_adult_gtex_v11_brain_caudate_basal_ganglia.gct.gz': '6f963630d66a16967afc66561ef9ae3c1c9175cc903df13e22bb6159bf45cf08', 'gene_reads_adult_gtex_v11_brain_cerebellar_hemisphere.gct.gz': 'f41d3b540baa99029b5510f1f4df1b7b5a6899f4249ac74e85471b7058821123', 'gene_reads_adult_gtex_v11_brain_cerebellum.gct.gz': 'ef380e98dd77483e897d4d53865af05ea3056d1f6a3c9daaf87dfea443587801', 'gene_reads_adult_gtex_v11_brain_cortex.gct.gz': '847f71ce8a4fd2aee2cf8079c9232c61235aebcd0ef896261846719bf3ada794', 'gene_reads_adult_gtex_v11_brain_frontal_cortex_ba9.gct.gz': '482dc7fdf872cb33bc608492b2c44f45def595ac2696c08d54acc09b82eeed1a', 'gene_reads_adult_gtex_v11_brain_hippocampus.gct.gz': 'f3b7c13d1c2638b88beb91e0860ee69573b9aecfa393ee8adc9722aa0b9e28d1', 'gene_reads_adult_gtex_v11_brain_hypothalamus.gct.gz': '809bba9ffd32f8ca83144391fa502ecd83635d96ff22f1a2f28fb2b650775d08', 'gene_reads_adult_gtex_v11_brain_nucleus_accumbens_basal_ganglia.gct.gz': 'b5afb55a6a3a70f15ad2be85ce88cfce403916a579cf12af9d29f2fa2a8c7446', 'gene_reads_adult_gtex_v11_brain_putamen_basal_ganglia.gct.gz': 'b156e492c467aef6cdc7f008a0be40fa08515e7b45008692c9d52ff028208922', 'gene_reads_adult_gtex_v11_brain_substantia_nigra.gct.gz': '08fe5e1b2fc3b0b138d608bd453596bdb856536b20d9dfbff969298451cceae3'}
DATA_URL = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}/releases/download/{RELEASE_TAG}/{ZIP_NAME}'

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

ZIP_PATH = BASE_DIR / 'gtex_downloads' / ZIP_NAME
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
if not ZIP_PATH.is_file() or file_sha256(ZIP_PATH) != EXPECTED_ZIP_SHA256:
    partial = ZIP_PATH.with_suffix('.download')
    print('Downloading:', DATA_URL)
    try:
        with urlopen(Request(DATA_URL, headers={'User-Agent':'BMEN600-GTEx-notebook'}), timeout=180) as response, partial.open('wb') as output:
            shutil.copyfileobj(response, output, 1024 * 1024)
    except Exception as exc:
        raise RuntimeError(f'Data download failed: {DATA_URL}. Check connectivity and release availability.') from exc
    if file_sha256(partial) != EXPECTED_ZIP_SHA256:
        raise ValueError('Downloaded ZIP checksum does not match the verified release. Do not use this download.')
    partial.replace(ZIP_PATH)
else:
    print('Using verified cached download.')

DATA_DIR = BASE_DIR / 'gtex_input' / EXPECTED_ZIP_SHA256[:12]
DATA_DIR.mkdir(parents=True, exist_ok=True)
expected_names = {f'gene_reads_adult_gtex_v11_brain_{region}.gct.gz' for region in BRAIN_REGIONS}
with zipfile.ZipFile(ZIP_PATH) as archive:
    selected = [member for member in archive.infolist()
                if not member.is_dir() and Path(member.filename).name in expected_names]
    found = [Path(member.filename).name for member in selected]
    if len(found) != len(set(found)):
        raise ValueError('Duplicate required brain filenames in archive.')
    missing = expected_names - set(found)
    if missing:
        raise ValueError('Missing brain files: ' + ', '.join(sorted(missing)))
    for member in selected:
        destination = DATA_DIR / Path(member.filename).name
        with archive.open(member) as src, destination.open('wb') as dst:
            shutil.copyfileobj(src, dst, 1024 * 1024)

paths = [DATA_DIR / name for name in sorted(expected_names)]
for path in paths:
    if file_sha256(path) != EXPECTED_FILE_SHA256[path.name]:
        raise ValueError('Brain-file checksum mismatch: ' + path.name)
print('Verified brain-region files:', len(paths))


## 3. Load, validate and normalize
A GCT 1.2 file has a version line, a dimensions line, then the table. `Name` identifies genes and `Description` supplies symbols. GTEx donor IDs are the first two hyphen-separated fields of the sample ID. Multiple samples for one donor within a region are averaged **after sample-level normalization**; the summary records this choice. We retain full Ensembl IDs to avoid collisions.

In [ ]:
def load_region(path):
    with gzip.open(path, 'rt') as f:
        version = f.readline().strip()
        n_genes, n_samples = map(int, f.readline().split())
    if version != '#1.2':
        raise ValueError(f'Unsupported GCT version: {version}')
    table = pd.read_csv(path, sep='\t', skiprows=2)
    if table.shape != (n_genes, n_samples + 2):
        raise ValueError(f'Dimension mismatch: {path.name}')
    if table['Name'].duplicated().any():
        raise ValueError('Duplicate gene IDs')
    symbols = table.set_index('Name')['Description']
    counts = table.iloc[:, 2:].astype('float32')
    counts.index = table['Name']
    if not np.isfinite(counts.to_numpy()).all() or (counts.to_numpy() < 0).any():
        raise ValueError('Counts must be finite and nonnegative')
    libraries = counts.sum(axis=0)
    if (libraries <= 0).any():
        raise ValueError('Zero-count sample found')
    donors = []
    for sample in counts.columns:
        parts = sample.split('-')
        if len(parts) < 3 or parts[0] != 'GTEX':
            raise ValueError(f'Unexpected sample ID: {sample}')
        donors.append('-'.join(parts[:2]))
    logcpm = np.log2(counts.div(libraries, axis=1) * 1e6 + 1).T
    logcpm.index = donors
    matrix = logcpm.groupby(level=0, sort=True).mean().astype('float32')
    record = dict(region=path.name.removeprefix('gene_reads_adult_gtex_v11_brain_').removesuffix('.gct.gz'),
                  genes=n_genes, samples=n_samples, donors=len(matrix),
                  extra_samples=n_samples-len(matrix), median_library_reads=float(libraries.median()),
                  sha256=hashlib.sha256(path.read_bytes()).hexdigest())
    return matrix, symbols, record

regions, symbols, records = {}, {}, []
for path in paths:
    matrix, gene_symbols, record = load_region(path)
    regions[record['region']] = matrix
    symbols[record['region']] = gene_symbols
    records.append(record)
    print(record['region'], matrix.shape)
summary = pd.DataFrame(records)
summary.to_csv(OUT / 'table_1_dataset_summary.csv', index=False)
display(summary.drop(columns='sha256'))
active = sorted(BRAIN_REGIONS)
overlap = pd.DataFrame([[len(regions[a].index.intersection(regions[b].index)) for b in active] for a in active], index=active, columns=active)
overlap.to_csv(OUT / 'donor_overlap.csv')
first_genes = regions[active[0]].columns
assert set(regions) == set(BRAIN_REGIONS)
assert all(regions[r].columns.equals(first_genes) for r in active), 'Gene identifiers/order differ'
print('Brain tissue records:', int(summary.samples.sum()))
print('Unique brain donors:', len(set().union(*(set(regions[r].index) for r in active))))


## 4. Dataset graphs
**Figure 1.** Unique donors per analyzed brain region after within-region aggregation. Counts describe available data, not the number used in every pair.

**Figure 2.** Donors with measurements in both regions. Each pair can have a different cohort. This limits direct comparison between pair scores.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
s = summary.sort_values('donors')
ax.barh(s.region.str.replace('_', ' '), s.donors, color='#357da8')
ax.set(xlabel='Unique donors', title='Figure 1. Available donors by region')
fig.tight_layout(); fig.savefig(OUT / 'figure_1_region_counts.png', dpi=200); plt.show()
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(overlap, cmap='Blues')
labels = [r.replace('_', ' ') for r in active]
ax.set_xticks(range(len(active)), labels, rotation=90)
ax.set_yticks(range(len(active)), labels)
for i in range(len(active)):
    for j in range(len(active)):
        ax.text(j, i, str(overlap.iloc[i,j]), ha='center', va='center', fontsize=7)
ax.set_title('Figure 2. Paired donor availability')
fig.colorbar(im, ax=ax, label='Paired donors')
fig.tight_layout(); fig.savefig(OUT / 'figure_2_donor_overlap.png', dpi=200); plt.show()

## 5. Prediction and evaluation
For each direction, hold out 20% of paired donors. Split the remaining development donors into inner training (75%) and validation (25%). Select genes using only inner training: CPM > 1 in at least 20% of donors, positive variance, and the highest variances. Source and target selection are separate. Keep these genes fixed during refitting on all development donors. Fit scaling independently at each training stage; choose ridge alpha from validation MSE. Test measurements do not guide tuning or selection.

The primary summary is **median gene-wise test R²**, where each gene's R² is `1 − Σ(y − ŷ)² / Σ(y − mean_test(y))²`. Negative values are retained, and near-constant test outcomes have undefined R² and are counted. The secondary **pooled skill** is `1 − model MSE / baseline MSE`, where the baseline predicts each gene's development-set mean. These statistics use different denominators and must not be interchanged. MAE and MSE are also exported.

Conditional intervals resample test donors with fixed predictions, preserving each donor's gene vector. They exclude training uncertainty. Different pair cohorts and target sets limit direct biological rankings. Expression means, variances, and regional mean-expression differences for the descriptive feature analysis are calculated only from development donors. There is no appended gene panel.


In [ ]:
def choose_genes(frame, n):
    expressed = (frame > 1).mean(axis=0) >= 0.2
    variance = frame.var(axis=0)
    ranked = variance[expressed & (variance > 1e-8)].sort_values(ascending=False)
    if len(ranked) < 10:
        raise ValueError('Too few expressed variable genes')
    return ranked.index[:n]

def analyze_pair(source, target):
    ids = regions[source].index.intersection(regions[target].index).sort_values()
    if len(ids) < MIN_PAIRED_DONORS:
        return None
    dev, test = train_test_split(np.arange(len(ids)), test_size=0.2, random_state=SEED)
    inner, val = train_test_split(dev, test_size=0.25, random_state=SEED)
    assert not (set(dev) & set(test)) and not (set(inner) & set(val))
    Xframe, Yframe = regions[source].loc[ids], regions[target].loc[ids]
    xgenes = choose_genes(Xframe.iloc[inner], N_SOURCE_GENES)
    ygenes = choose_genes(Yframe.iloc[inner], N_TARGET_GENES)
    X = Xframe[xgenes].to_numpy(dtype='float64')
    Y = Yframe[ygenes].to_numpy(dtype='float64')
    scaler = StandardScaler().fit(X[inner])
    Xi, Xv = scaler.transform(X[inner]), scaler.transform(X[val])
    validation = []
    for alpha in ALPHAS:
        fit = Ridge(alpha=alpha, solver='cholesky').fit(Xi, Y[inner])
        validation.append(np.mean((Y[val] - fit.predict(Xv))**2))
    best = ALPHAS[int(np.argmin(validation))]
    scaler = StandardScaler().fit(X[dev])
    model = Ridge(alpha=best, solver='cholesky').fit(scaler.transform(X[dev]), Y[dev])
    truth = Y[test]
    pred = model.predict(scaler.transform(X[test]))
    baseline = np.broadcast_to(Y[dev].mean(axis=0), truth.shape)
    model_loss = ((truth-pred)**2).mean(axis=1)
    base_loss = ((truth-baseline)**2).mean(axis=1)
    skill = 1-model_loss.mean()/base_loss.mean()
    rng = np.random.default_rng(SEED)
    sampled = rng.integers(0, len(test), size=(N_BOOTSTRAPS, len(test)))
    bs = 1-model_loss[sampled].mean(axis=1)/base_loss[sampled].mean(axis=1)
    lo, hi = np.quantile(bs, [0.025, 0.975])
    sse = ((truth-pred)**2).sum(axis=0)
    sst = ((truth-truth.mean(axis=0))**2).sum(axis=0)
    r2 = np.divide(sse, sst, out=np.full_like(sse, np.nan), where=sst>1e-12)
    gene_table = pd.DataFrame({'gene_id':ygenes, 'symbol':symbols[target].reindex(ygenes).values,
                               'test_r2':1-r2, 'model_mse':((truth-pred)**2).mean(axis=0),
                               'baseline_mse':((truth-baseline)**2).mean(axis=0),
                               'target_dev_mean':Yframe.iloc[dev][ygenes].mean().values,
                               'target_dev_variance':Yframe.iloc[dev][ygenes].var().values,
                               'source_dev_mean':Xframe.iloc[dev].reindex(columns=ygenes).mean().values})
    gene_table['abs_regional_mean_difference'] = abs(gene_table.target_dev_mean-gene_table.source_dev_mean)
    gene_table['gene_skill'] = 1-gene_table.model_mse/gene_table.baseline_mse
    gene_table['source'], gene_table['target'] = source, target
    # Conditional bootstrap of median gene-wise R², resampling donors, not genes.
    r2_boot = []
    for sample in sampled:
        t, pr = truth[sample], pred[sample]
        den = ((t-t.mean(axis=0))**2).sum(axis=0)
        rr = 1-np.divide(((t-pr)**2).sum(axis=0),den,out=np.full_like(den,np.nan),where=den>1e-12)
        r2_boot.append(np.nanmedian(rr))
    r2lo,r2hi = np.nanquantile(r2_boot,[0.025,0.975])
    prefix = source+'__to__'+target
    gene_table.to_csv(OUT / (prefix+'_genes.csv'), index=False)
    pd.DataFrame({'gene_id':xgenes, 'symbol':symbols[source].reindex(xgenes).values}).to_csv(OUT / (prefix+'_source_genes.csv'), index=False)
    # Keep donor-level files local; avoid sharing IDs in public repositories.
    pd.DataFrame({'donor_id':ids[test], 'model_mse':model_loss, 'baseline_mse':base_loss}).to_csv(OUT / (prefix+'_test_errors_PRIVATE.csv'), index=False)
    pd.DataFrame({'donor_id':ids, 'split':['test' if i in test else 'development' for i in range(len(ids))]}).to_csv(OUT / (prefix+'_splits_PRIVATE.csv'), index=False)
    row = dict(source=source,target=target,n_paired=len(ids),n_development=len(dev),n_test=len(test),
               n_source_genes=len(xgenes),n_target_genes=len(ygenes),alpha=best,
               model_mse=model_loss.mean(),baseline_mse=base_loss.mean(),
               skill=skill,ci_low=lo,ci_high=hi,median_gene_r2=float(np.nanmedian(1-r2)),
               r2_ci_low=float(r2lo),r2_ci_high=float(r2hi),
               mean_absolute_error=float(np.mean(np.abs(truth-pred))),
               undefined_gene_r2_count=int(np.isnan(1-r2).sum()))
    return row, dict(truth=truth,pred=pred,baseline=baseline,genes=gene_table)

pairs = list(itertools.permutations(active, 2)) if RUN_ALL_PAIRS else ILLUSTRATIVE_PAIRS
results, skipped, example, gene_results = [], [], None, []
for source,target in pairs:
    if source not in active or target not in active:
        raise ValueError('Selected region unavailable or excluded')
    output = analyze_pair(source,target)
    if output is None:
        skipped.append({'source':source,'target':target,'reason':'too few paired donors'})
        continue
    row, detail = output
    results.append(row)
    gene_results.append(detail['genes'])
    if example is None:
        example = (row,detail)
    print(source,'→',target, 'skill:',round(row['skill'],3))
if not results:
    raise ValueError('No eligible pairs. Inspect overlap and minimum donor threshold.')
results = pd.DataFrame(results)
results.to_csv(OUT / 'table_2_prediction_results.csv',index=False)
pd.DataFrame(skipped,columns=['source','target','reason']).to_csv(OUT / 'skipped_pairs.csv', index=False)
display(results)
all_genes = pd.concat(gene_results,ignore_index=True)
all_genes.to_csv(OUT / 'table_3_gene_transferability_features.csv',index=False)

## 6. Result graphs
**Figure 3.** Prediction skill with conditional 95% donor-bootstrap intervals. Zero indicates the mean-expression baseline. An interval crossing zero does not show clear improvement for this held-out cohort.

**Figure 4.** Predicted versus observed expression for the example target gene with greatest variance in the inner-training donors (not selected for best test performance). Each point is one test donor. The dashed line is perfect prediction.

**Figure 5 (all-pair mode).** Directional median gene-wise test R². Rows are sources, columns are targets; missing pairs remain blank. Pairwise rankings require the sensitivity analyses described above.

In [ ]:
r = results.sort_values('skill')
fig, ax = plt.subplots(figsize=(10,max(3,0.3*len(r))))
y = np.arange(len(r))
ax.hlines(y,r.ci_low,r.ci_high,color='#357da8')
ax.scatter(r.skill,y,color='#163b54')
ax.set_yticks(y, [a.replace('_',' ')+' → '+b.replace('_',' ') for a,b in zip(r.source,r.target)])
ax.axvline(0,color='gray',linestyle='--')
ax.set(xlabel='Skill relative to target-mean baseline',title='Figure 3. Cross-region prediction skill (95% intervals)')
fig.tight_layout(); fig.savefig(OUT / 'figure_3_prediction_skill.png',dpi=200); plt.show()
row, detail = example
# Target genes are ordered by development-inner-training variance.
j = 0
truth, pred = detail['truth'][:,j], detail['pred'][:,j]
fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(truth,pred,alpha=0.8,color='#357da8')
limits = [min(truth.min(),pred.min()),max(truth.max(),pred.max())]
ax.plot(limits,limits,'--',color='gray')
name = detail['genes'].iloc[j]['symbol']
ax.set(xlabel='Observed log2(CPM + 1)',ylabel='Predicted log2(CPM + 1)',title=f'Figure 4. Held-out donors: {name}')
fig.tight_layout(); fig.savefig(OUT / 'figure_4_example_gene.png',dpi=200); plt.show()
if RUN_ALL_PAIRS:
    heat = results.pivot(index='source',columns='target',values='median_gene_r2').reindex(index=active,columns=active)
    fig, ax = plt.subplots(figsize=(11,9))
    bound = max(1,float(np.nanmax(np.abs(heat.to_numpy()))))
    im = ax.imshow(np.ma.masked_invalid(heat.to_numpy()),cmap='RdBu',vmin=-bound,vmax=bound)
    ax.set_xticks(range(len(active)),labels,rotation=90)
    ax.set_yticks(range(len(active)),labels)
    ax.set(xlabel='Target',ylabel='Source',title='Figure 5. Regional transcriptomic predictability')
    fig.colorbar(im,ax=ax,label='Median gene-wise test R²')
    fig.tight_layout(); fig.savefig(OUT / 'figure_5_transferability_heatmap.png',dpi=200); plt.show()

## 7. Molecular characteristics associated with transferability
**Figure 6:** Median gene-wise test R² and conditional 95% donor-bootstrap intervals by direction.

**Figure 7:** For the first evaluated pair, gene-wise test R² versus development-set regional mean-expression difference, coloured by target expression level. The CSV also reports within-pair Spearman associations for expression level, variance and regional difference. These are descriptive associations among correlated genes, with no independent-gene significance claim; expression variance and R² are mathematically related and may reflect measurement reliability.



In [ ]:
# Pair-level R² overview
r = results.sort_values('median_gene_r2')
fig,ax = plt.subplots(figsize=(10,max(3,0.35*len(r))))
y = np.arange(len(r))
ax.hlines(y,r.r2_ci_low,r.r2_ci_high,color='#357da8')
ax.scatter(r.median_gene_r2,y,color='#163b54')
ax.set_yticks(y,[a.replace('_',' ')+' → '+b.replace('_',' ') for a,b in zip(r.source,r.target)])
ax.axvline(0,color='gray',linestyle='--')
ax.set(xlabel='Median gene-wise test R²',title='Figure 6. Transferability across region pairs')
fig.tight_layout();fig.savefig(OUT/'figure_6_gene_r2.png',dpi=200);plt.show()

features = ['target_dev_mean','target_dev_variance','abs_regional_mean_difference']
associations=[]
for (source,target), group in all_genes.groupby(['source','target']):
    for feature in features:
        valid=group[[feature,'test_r2']].replace([np.inf,-np.inf],np.nan).dropna()
        rho=valid[feature].rank().corr(valid.test_r2.rank()) if len(valid)>2 else np.nan
        associations.append(dict(source=source,target=target,feature=feature,n_genes=len(valid),spearman_rho=rho))
associations=pd.DataFrame(associations)
associations.to_csv(OUT/'table_4_molecular_associations.csv',index=False)
display(associations)
first=all_genes[(all_genes.source==results.iloc[0].source)&(all_genes.target==results.iloc[0].target)]
fig,ax=plt.subplots(figsize=(7,5))
im=ax.scatter(first.abs_regional_mean_difference,first.test_r2,c=first.target_dev_mean,cmap='viridis',s=15,alpha=0.7)
ax.axhline(0,color='gray',linestyle='--')
ax.set(xlabel='Absolute regional difference in mean log2(CPM + 1)',ylabel='Gene-wise test R²',title='Figure 7. Molecular characteristics and predictability')
fig.colorbar(im,ax=ax,label='Target development mean log2(CPM + 1)')
fig.tight_layout();fig.savefig(OUT/'figure_7_molecular_features.png',dpi=200);plt.show()



## 8. Interpretation and reporting
Use the dataset summary and Figures 1–2 for brain-region coverage. Table 2 contains the six-direction pilot results; Figure 6 shows the primary median R² and Figure 3 shows pooled skill. Figure 7 and Table 4 describe molecular-feature associations. These exploratory correlations do not establish causal relationships, and overlapping donors prevent treating pair scores as independent cases.

This analysis concerns predictable variation in selected bulk-tissue measurements. Age, sex, tissue quality, cell mixture, and batch may affect scores. Before the final report, repeat complete donor splits across seeds 42–46; examine common genes/cohorts, available covariates, and normalization sensitivity; and run the full map. None of those extensions is completed by the default pilot run.

Record original GTEx provenance and the submitted repository commit. Input hashes, exact parameters, selected genes, aggregate outputs, and package versions are exported for each run. Donor-level files remain outside the public ZIP. Review applicable sharing terms before publishing data.

AI-use statement to adapt truthfully: ChatGPT assisted with code review, corrections, local execution, and reporting. Team members must independently review the methods and references, rerun the analysis, and record their checks before submission.


In [ ]:
metadata = {'question':'Brain-region transcriptomic prediction in unseen GTEx donors',
            'seed':SEED, 'run_id':RUN_ID, 'run_all_pairs':RUN_ALL_PAIRS,
            'brain_regions':BRAIN_REGIONS, 'primary_metric':'median gene-wise test R²',
            'normalization':'log2(CPM+1); total counts per sample before gene selection',
            'source_genes':N_SOURCE_GENES, 'target_genes':N_TARGET_GENES,
            'alphas':ALPHAS, 'bootstraps':N_BOOTSTRAPS,
            'python':platform.python_version(), 'numpy':np.__version__,
            'pandas':pd.__version__, 'sklearn':sklearn.__version__,
            'data_url':DATA_URL, 'release_zip_sha256':EXPECTED_ZIP_SHA256,
            'input_files':records, 'illustrative_pairs':ILLUSTRATIVE_PAIRS}
(OUT/'analysis_metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
# Unique run directories keep prior runs out of the public export.
PUBLIC = OUT.parent / ('public_' + RUN_ID)
PUBLIC.mkdir(exist_ok=False)
for p in OUT.iterdir():
    if p.is_file() and 'PRIVATE' not in p.name:
        shutil.copy2(p,PUBLIC/p.name)
archive = shutil.make_archive(str(PUBLIC),'zip',PUBLIC)
from google.colab import files
files.download(archive)


## 9. References

[1] M. J. Hawrylycz et al., “An anatomically comprehensive atlas of the adult human brain transcriptome,” *Nature*, vol. 489, pp. 391–399, 2012. doi: [10.1038/nature11405](https://doi.org/10.1038/nature11405).

[2] M. Hawrylycz et al., “Canonical genetic signatures of the adult human brain,” *Nature Neuroscience*, vol. 18, pp. 1832–1844, 2015. doi: [10.1038/nn.4171](https://doi.org/10.1038/nn.4171).

[3] GTEx Consortium, “The GTEx Consortium atlas of genetic regulatory effects across human tissues,” *Science*, vol. 369, pp. 1318–1330, 2020. doi: [10.1126/science.aaz1776](https://doi.org/10.1126/science.aaz1776).

[4] A. E. Hoerl and R. W. Kennard, “Ridge regression: Biased estimation for nonorthogonal problems,” *Technometrics*, vol. 12, no. 1, pp. 55–67, 1970. doi: [10.1080/00401706.1970.10488634](https://doi.org/10.1080/00401706.1970.10488634).

[5] S. Varma and R. Simon, “Bias in error estimation when using cross-validation for model selection,” *BMC Bioinformatics*, vol. 7, art. 91, 2006. doi: [10.1186/1471-2105-7-91](https://doi.org/10.1186/1471-2105-7-91).

[6] M. D. Robinson and A. Oshlack, “A scaling normalization method for differential expression analysis of RNA-seq data,” *Genome Biology*, vol. 11, R25, 2010. doi: [10.1186/gb-2010-11-3-r25](https://doi.org/10.1186/gb-2010-11-3-r25).

[7] J. Ke, T. Ge, and R. D. Melamed, “Discovering disease genetic variation impacting gene expression in 103 brain tissues with the Brain Ontology Expression (BRONTE) graph neural network model,” *bioRxiv*, preprint, May 2025. doi: [10.1101/2025.05.08.652857](https://doi.org/10.1101/2025.05.08.652857).

[8] GTEx Portal, “Datasets,” Adult GTEx V11 release documentation. Accessed: Oct. 15, 2026. [Online]. Available: https://gtexportal.org/home/datasets/
